# Q-Learning（在线策略版本 On-Policy）

> Q-Learning 是一种 **Off-Policy TD 控制**算法，其核心创新在于：将更新目标中的「实际下一步动作 $a_{t+1}$」替换为「下一状态下贪心最优动作的 Q 值」，即 $\max_{a'} Q(s_{t+1}, a')$，从而使**目标策略**（Greedy）与**行为策略**（$\varepsilon$-Greedy）相解耦。
>
> **SARSA** 的更新公式（目标依赖实际采样到的 $a_{t+1}$，On-Policy）：
>
> $$Q(s_t,a_t) \leftarrow Q(s_t,a_t) - \alpha \underbrace{\bigl[Q(s_t,a_t) - (r_{t+1} + \gamma Q(s_{t+1},a_{t+1}))\bigr]}_{\text{TD 误差，用 }(s,a,r,s',a')}$$
>
> **Q-Learning** 的更新公式（目标改为贪心最优，与行为策略无关）：
>
> $$Q(s_t,a_t) \leftarrow Q(s_t,a_t) - \alpha \left[Q(s_t,a_t) - \left(r_{t+1} + \gamma \max_{a'} Q(s_{t+1},a')\right)\right]$$
>
> 两者的核心对比如下：
>
> ---

| 对比维度 | SARSA | Q-Learning |
|---|---|---|
| 更新目标 | 实际执行的 $Q(s_{t+1},a_{t+1})$ | $\max_{a'} Q(s_{t+1},a')$ 贪心最优值 |
| **策略类型** | **On-Policy**（行为策略 = 目标策略） | **Off-Policy**（行为策略 ≠ 目标策略） |
| 风险偏好 | 保守（绕开危险区域） | 激进（沿最优路径，容忍风险） |
| 收敛策略 | 收敛到 $\varepsilon$-贪心最优策略 | 收敛到真正最优策略 |
| 所需样本 | $(s,a,r,s',a')$ 五元组 | $(s,a,r,s')$ 四元组 |

> ---
>
> **本版本（On-Policy Q-Learning）**：将行为策略也设为 $\varepsilon$-Greedy，使行为策略与目标策略保持一致，构成 On-Policy 变体。更新目标仍为 $\max_{a'} Q(s_{t+1},a')$，与标准 SARSA 的本质区别不变。

> **行为策略与目标策略**是理解 On/Off-Policy 的关键抓手。任何强化学习控制算法都同时维护两套策略：
>
> - **行为策略 $\pi_b$**：直接与环境交互、负责采样经验，通常加入随机性以保证探索（对应代码中的 `policy_epsilon`）
> - **目标策略 $\pi_T$**：被持续更新、最终希望收敛为最优策略，通常为贪心或确定性策略（对应代码中的 `policy`）
>
> 两套策略是否相同决定了算法的同/异策略属性，对比如下：
>
> ---

| 策略类型 | 定义 | 本 Notebook 例子 |
|---|---|---|
| **同策略（On-Policy）** | $\pi_b$ 与 $\pi_T$ **相同**，用同一策略采样并更新 | SARSA、Expected SARSA、本版本 Q-Learning |
| **异策略（Off-Policy）** | $\pi_b$ 与 $\pi_T$ **可以不同**，行为策略探索、目标策略贪心 | 标准 Q-Learning |

> ---
>
> Q-Learning 的更新目标 $\max_{a'} Q(s_{t+1},a')$ 始终对应**贪心目标策略**，因此即使行为策略采用 $\varepsilon$-Greedy（如本版本），两者在数学上仍属不同策略，Q-Learning 本质仍是 Off-Policy 思路；本版本只是将两者「对齐」为同一 $\varepsilon$-Greedy 策略，形成工程上的 On-Policy 变体。
>
> 两者在**数据复用能力**上也存在本质差异：
> - **同策略**：更新目标与当前正在优化的 $\pi$ 深度绑定，旧策略或外部策略采集的样本目标错位，**只能依赖在线实时数据**；
> - **异策略**：更新不依赖「行为策略实际选取了哪个 $a'$」，历史经验可反复利用，**既支持在线训练，也天然支持离线经验回放**（Experience Replay）。

## 一、导入库与环境初始化

In [1]:
# 导入 NumPy 库，用于数组操作（Q 表、策略矩阵等）
import numpy as np
# 导入 random 模块，用于随机选择初始状态和动作
import random
# 导入 importlib 标准库，用于按文件路径动态加载模块
import importlib.util
# 导入 os 模块，用于拼接当前目录与文件名
import os
# 导入 time 模块（备用，可控制训练展示节奏）
import time
# 导入 Jupyter 显示控制函数，训练时清除输出以实时刷新展示
from IPython.display import clear_output

# 环境文件名 "02.1.ModelFree_Env_GridWorldV2.py" 以数字开头，
# Python 无法直接 import，需通过 importlib 按路径加载
_env_path = os.path.join(
    os.path.dirname(os.path.abspath("__file__")),
    "02.1.ModelFree_Env_GridWorldV2.py")  # 环境文件的绝对路径（str）
_spec = importlib.util.spec_from_file_location(
    "GridWorld_v2", _env_path)   # 构造模块规格对象
GridWorld_v2 = importlib.util.module_from_spec(_spec)  # 根据规格创建模块对象
_spec.loader.exec_module(GridWorld_v2)                  # 执行模块代码，完成加载

In [2]:
# 设置网格世界行数（必须与 desc 字符串行数一致）
rows = 5
# 设置网格世界列数（必须与 desc 每行字符数一致）
columns = 5

# 创建 5×5 GridWorld 环境实例
# forbiddenAreaScore=-10: 进入障碍格（'#'）的即时惩罚
# score=1: 到达目标格（'T'）的即时正奖励
# desc: 地图描述，'.'=普通格，'#'=障碍格，'T'=目标格
gridworld = GridWorld_v2.GridWorld_v2(
    forbiddenAreaScore=-10, score=1,
    desc=[".....", ".##..", "..#..", ".#T#.", ".#..."])

# 显示网格地图
gridworld.show()

# 初始化状态价值向量（全零，用于展示）
# shape: (25,)——每个元素对应一个状态（0~24）的价值估计
value = np.zeros(rows * columns)

# 初始化 Q 表（动作价值矩阵，全零）
# shape: (25, 5)——行=状态索引[0~24]，列=5种动作的 Q 值
qtable = np.zeros((rows * columns, 5))

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️


## 二、Q-Learning On-Policy 算法实现

In [3]:
def QLearning_On_Policy(gridworld: GridWorld_v2.GridWorld_v2,
                        gamma=0.99,
                        trajectorySteps=-1,
                        learning_rate=0.001,
                        final_epsilon=0.01,
                        num_episodes=600) -> GridWorld_v2.GridWorld_v2:
    '''
    Q-Learning 算法（On-Policy 版本）。

    使用 epsilon-greedy 策略同时采样和更新 Q 值，
    更新目标使用下一状态的最优 Q 值 max Q(s',a')，
    而非实际执行的动作的 Q 值（区别于 SARSA）。

    参数:
        gridworld (GridWorld_v2): GridWorld 环境实例
        gamma (float): 折扣因子，取值 (0,1]
        trajectorySteps (int): 每条轨迹最大步数；-1 表示到目标自动停止
        learning_rate (float): TD 学习率，控制 Q 值更新幅度
        final_epsilon (float): epsilon 的最小值，防止完全停止探索
        num_episodes (int): 训练轮数

    返回值:
        GridWorld_v2.GridWorld_v2: 训练后的环境对象（可查看最终策略）
    '''
    # 初始化状态价值向量（全零，仅用于展示）
    # shape: (25,)——每个元素对应一个状态的价值
    state_value = np.zeros((rows * columns))

    # 初始化动作价值矩阵 Q（全零）
    # shape: (25, 5)——行=状态[0~24]，列=5种动作的 Q 值
    action_value = np.zeros((rows * columns, 5))

    # 初始化随机确定性策略（独热编码）
    # shape: (25, 5)——每行只有一个位置为 1（对应随机选取的初始动作）
    policy = np.eye(5)[np.random.randint(0, 5, size=(rows * columns))]

    epsilon = 0.5  # 初始 epsilon（较高，鼓励早期探索）

    for episode in range(num_episodes):  # 训练主循环
        print("episode", f"{episode}/{num_episodes}")  # 打印当前轮次

        # 线性衰减 epsilon（逐步从探索转向利用）
        if epsilon > final_epsilon:
            epsilon -= 0.001   # 每轮减少 0.001
        else:
            epsilon = final_epsilon  # 不低于最小探索率

        # 计算 epsilon-greedy 概率分配
        # p1：贪心动作的选取概率 = 1 - ε*(4/5)
        p1 = 1 - epsilon * (4/5)
        # p0：非贪心动作各自的选取概率 = ε/5
        p0 = epsilon / 5
        # 概率映射字典：策略矩阵中 1→p1，0→p0
        d = {1: p1, 0: p0}
        print("p1", p1, "p0", p0)  # 打印当前概率值

        # 将确定性策略（独热）转为 epsilon-greedy 概率分布
        # shape: (25, 5)——每行是一个状态的动作概率分布
        policy_epsilon = np.vectorize(d.get)(policy)

        # 初始化状态访问计数器（长度 25，记录各状态访问次数）
        cnt = [0 for i in range(25)]

        # 固定初始状态为 10（地图中部）
        initState = 10
        # 随机选择初始动作（0~4 的整数）
        initAction = random.randint(0, 4)

        if trajectorySteps == -1:
            stop_when_reach_target = True  # 到达目标时自动停止

        # 按 epsilon-greedy 策略生成一条轨迹
        # 返回值：列表，每个元素为 (状态, 动作, 奖励, 下一状态, 下一动作) 五元组
        Trajectory = gridworld.getTrajectoryScore(
            nowState=initState,
            action=initAction,
            policy=policy_epsilon,
            steps=trajectorySteps,
            stop_when_reach_target=True)

        print("trajectorySteps", len(Trajectory))  # 打印轨迹长度

        steps = len(Trajectory) - 1  # 有效步数

        # 从轨迹末尾向前遍历，执行 Q-Learning 的 TD 更新
        for k in range(steps, -1, -1):
            # 解包当前步转移
            # tmpstate:   当前状态 int
            # tmpaction:  当前动作 int
            # tmpscore:   即时奖励 float
            # nextState:  下一状态 int
            # nextAction: 下一动作 int（Q-Learning 中不使用！）
            tmpstate, tmpaction, tmpscore, nextState, nextAction = Trajectory[k]
            cnt[tmpstate] += 1  # 记录状态访问次数

            # Q-Learning TD 误差（与 SARSA 的关键区别）
            # Q-Learning：使用 max Q(s', a')——下一状态所有动作中最优的 Q 值
            # SARSA：使用 Q(s', a')——实际执行动作的 Q 值
            # action_value[nextState].max()：shape 标量 float，下一状态的最大 Q 值
            TD_error = (action_value[tmpstate][tmpaction]
                        - (tmpscore + gamma * action_value[nextState].max()))
            # 按 TD 误差更新 Q 值
            action_value[tmpstate][tmpaction] -= learning_rate * TD_error

        # 策略改进：每个状态选 Q 值最大的动作作为新贪心策略
        # np.argmax(action_value, axis=1)：shape (25,)，每个状态最优动作索引
        # np.eye(5)[...]：转为独热编码，shape (25, 5)
        policy = np.eye(5)[np.argmax(action_value, axis=1)]
        # 更新 epsilon-greedy 策略供下轮使用
        policy_epsilon = np.vectorize(d.get)(policy)

        print(np.array(cnt).reshape(5, 5))  # 打印 5×5 状态访问次数矩阵

        # 计算状态价值 V(s) = Σ_a π(a|s)*Q(s,a)，shape (25,)
        state_value = np.sum(policy_epsilon * action_value, axis=1)
        # 全局平均状态价值（衡量整体策略质量）
        mean_state_value = np.sum(policy_epsilon * action_value, axis=1).mean()

        gridworld.showPolicy(policy)                              # 显示当前策略
        print(np.round(state_value, decimals=4).reshape(5, 5))   # 打印 5×5 状态价值
        print("mean_state_value", mean_state_value)              # 打印平均状态价值

    return gridworld  # 返回训练后的环境对象

## 三、运行 Q-Learning On-Policy 并查看结果

In [4]:
# 调用 Q-Learning On-Policy 算法训练智能体
# Q-Learning 使用最优 Q 值更新（max），通常比 SARSA 收敛更快、效果更好
# 直接可得全局最优策略（而非 SARSA 可能停留于次优）
QLearning_On_Policy(gridworld)

episode 0/600
p1 0.6008 p0 0.0998
trajectorySteps 5
[[0 0 0 0 0]
 [0 0 0 0 0]
 [2 0 0 0 0]
 [1 1 1 0 0]
 [0 0 0 0 0]]
⬆️⬆️⬆️⬆️⬆️
⬆️⏫️⏫️⬆️⬆️
⬆️⬆️⏫️⬆️⬆️
⬆️⏩️✅⏫️⬆️
⬆️⏫️⬆️⬆️⬆️
[[ 0.      0.      0.      0.      0.    ]
 [ 0.      0.      0.      0.      0.    ]
 [-0.0001  0.      0.      0.      0.    ]
 [-0.001   0.0006 -0.001   0.      0.    ]
 [ 0.      0.      0.      0.      0.    ]]
mean_state_value -5.979604792000001e-05
episode 1/600
p1 0.6015999999999999 p0 0.0996
trajectorySteps 373
[[31 55 68 99 67]
 [ 7 12  9 13  4]
 [ 2  2  2  1  0]
 [ 0  0  1  0  0]
 [ 0  0  0  0  0]]
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬇️⏬⬆️⬆️
⬆️⏩️✅⏫️⬆️
⬆️⏫️⬆️⬆️⬆️
[[-0.0023 -0.01   -0.0092 -0.0053 -0.0045]
 [-0.0011 -0.002  -0.003  -0.001   0.    ]
 [-0.0001 -0.002  -0.0004 -0.001   0.    ]
 [-0.001   0.0006 -0.002   0.      0.    ]
 [ 0.      0.      0.      0.      0.    ]]
mean_state_value -0.0017683473742979867
episode 2/600
p1 0.6024 p0 0.0994
trajectorySteps 5
[[0 0 0 0 0]
 [0 0 0 0 0]
 [2 1 0 0 0]
 [0 1 1 0 0]
 [